# Capítulo 10: Naive Bayes

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 13 de Grus (2019).

> É bom para o coração ser ingênuo, e para a mente não ser.
>
> — Anatole France

Uma rede social não é grande coisa se as pessoas não puderem se comunicar por ela. Por isso a DataSciencester tem uma funcionalidade popular de troca de mensagens entre membros — e, como em qualquer lugar onde pessoas trocam mensagens, alguns poucos usuários mal-intencionados insistem em enviar spam para os outros: esquemas de enriquecimento rápido, remédios sem receita, cursos de certificação em ciência de dados. Os usuários começaram a reclamar, e o VP de Mensageria pediu que você usasse ciência de dados para encontrar um jeito de filtrar essas mensagens.

Este capítulo faz duas coisas ao mesmo tempo. A primeira é apresentar o **Naive Bayes**, um classificador probabilístico que se apoia inteiramente no teorema de Bayes e numa suposição de independência tão grosseira que dá nome ao método. A segunda é completar o vocabulário do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html): lá, a seção sobre atributos prometeu que o Naive Bayes é o classificador certo quando os seus atributos são do tipo sim-ou-não. Este capítulo cumpre a promessa, construindo um modelo em que cada palavra do vocabulário *é* um atributo desses — presente ou ausente, nada entre os dois.

O [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html) já mostrou um classificador construído do zero, testado sobre dados reais, terminando numa matriz de confusão. Este capítulo repete essa forma, mas com um modelo de natureza oposta: onde o k-vizinhos não treina — guarda o conjunto de dados inteiro e faz a conta na hora de prever —, o Naive Bayes treina de fato, percorrendo as mensagens de treino uma vez para acumular contagens, e depois nunca mais olha para os dados brutos.

Este também é o primeiro modelo probabilístico do livro: `predict` devolve uma probabilidade, não um rótulo, e cabe a quem usa o modelo decidir o limiar de decisão — escolha que reaparece, com outro nome, na regressão logística. E é o primeiro capítulo em que treinar significa comprimir os dados de treino em alguns poucos parâmetros — aqui, um vocabulário e dois arrays de contagens alinhados a ele — e descartar o resto. É a postura que todo modelo a partir daqui assume, incluindo os coeficientes que a sequência de regressão, começando no próximo capítulo, vai ajustar por gradiente descendente.

Ao final deste capítulo, você será capaz de:

- Explicar a suposição de independência que dá nome ao Naive Bayes, e por que ela funciona bem mesmo sendo irrealista
- Justificar, com um exemplo numérico, por que somar log-probabilidades evita o *underflow* de multiplicar muitas probabilidades pequenas
- Explicar por que a suavização por pseudocontador é necessária, e o que dá errado sem ela
- Implementar um classificador Naive Bayes do zero: tokenizar, contar, e prever comparando a mensagem com o vocabulário inteiro por uma máscara booleana
- Reconhecer, num `assert`, o erro clássico de comparar `float` por igualdade exata
- Avaliar o classificador com as métricas do Capítulo 8, e explicar por que um conjunto desbalanceado exige mais do que acurácia
- Reconhecer o mesmo algoritmo na interface do `scikit-learn`

## Seções

| Seção | Tópico |
|---|---|
| [10.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/01-um-filtro-de-spam-bem-burro.html) | Um Filtro de Spam Bem Burro |
| [10.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-um-filtro-mais-sofisticado.html) | Um Filtro de Spam Mais Sofisticado |
| [10.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/03-implementacao.html) | Implementação |
| [10.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/04-testando-o-modelo.html) | Testando o Modelo |
| [10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html) | Usando o Modelo |

## Um Filtro de Spam Bem Burro

> **📌 Nota**
>
> Esta seção corresponde a *A Really Dumb Spam Filter*, do capítulo 13 de Grus (2019).

Ao longo deste capítulo, chamamos as mensagens legítimas — as que não são spam — de **ham**. É jargão em inglês, uma piada com *spam*: os dois são nomes de carne enlatada, e a dupla pegou emprestado o sentido de "mensagem indesejada" de um esquete do Monty Python. Fica só o nome; não precisa gostar de presunto para entender o capítulo.

Imagine um "universo" que consiste em receber uma mensagem escolhida aleatoriamente dentre todas as mensagens possíveis. De 1.000 mensagens desse universo, digamos que 500 são spam e 500 são ham. Das 500 de spam, 250 contêm a palavra *bitcoin*. Das 500 ham, só 5 contêm.

Quem recebe uma mensagem com *bitcoin* está diante de $250 + 5 = 255$ mensagens possíveis — e 250 delas são spam. A probabilidade de que a mensagem seja spam, dado que ela contém *bitcoin*, é simplesmente:

$$
\frac{250}{255} \approx 98\%
$$

Essa é a conta inteira. O resto desta seção é só escrever a mesma conta em símbolos, para poder generalizá-la depois.

Seja $S$ o evento "a mensagem é spam" e $B$ o evento "a mensagem contém a palavra *bitcoin*". O teorema de Bayes diz que:

$$
P(S \mid B) = \frac{P(B \mid S)\, P(S)}{P(B \mid S)\, P(S) + P(B \mid \lnot S)\, P(\lnot S)}
$$

onde $\lnot S$ é o evento complementar de $S$ — "a mensagem não é spam". O numerador é a probabilidade de a mensagem ser spam **e** conter *bitcoin*; o denominador é a probabilidade de a mensagem conter *bitcoin*, seja ela spam ou não. Ou seja: essa fração é exatamente a proporção de mensagens com *bitcoin* que são spam — a mesma pergunta que a conta de contagem acima já respondeu.

Se tivermos uma grande coleção de mensagens que sabemos serem spam e uma grande coleção que sabemos não serem, dá para estimar $P(B \mid S)$ e $P(B \mid \lnot S)$ contando quantas mensagens de cada grupo contêm a palavra — exatamente como fizemos acima: $P(B \mid S) = 250/500 = 0{,}5$ e $P(B \mid \lnot S) = 5/500 = 0{,}01$. E se, além disso, assumirmos que qualquer mensagem tem a mesma chance de ser spam ou não ($P(S) = P(\lnot S) = 0{,}5$) — o que já vale no nosso exemplo de 1.000 mensagens, mas é uma suposição adicional em geral —, a expressão simplifica para:

$$
P(S \mid B) = \frac{P(B \mid S)}{P(B \mid S) + P(B \mid \lnot S)}
$$

Por exemplo: se 50% das mensagens de spam contêm a palavra *bitcoin*, mas só 1% das mensagens legítimas contêm, a probabilidade de que uma mensagem qualquer contendo *bitcoin* seja spam é:

In [ ]:
p_bitcoin_dado_spam = 0.5
p_bitcoin_dado_ham = 0.01

p_spam_dado_bitcoin = p_bitcoin_dado_spam / (p_bitcoin_dado_spam + p_bitcoin_dado_ham)
p_spam_dado_bitcoin

98% — a mesma fração de sempre, agora com proporções em vez de contagens absolutas.

> **🔷 Conceito**
>
> Esse é o filtro "bem burro" do título: uma única palavra decide tudo. Ele já captura a ideia inteira do Naive Bayes — inverter, via Bayes, uma probabilidade fácil de estimar por contagem (a chance de *bitcoin* aparecer, dado que já sabemos se a mensagem é spam) para responder à pergunta que realmente importa (a chance de a mensagem ser spam, dado que ela contém *bitcoin*).
>
> O que falta é generalizar de uma palavra para um vocabulário inteiro. É o assunto da [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-um-filtro-mais-sofisticado.html).

## Um Filtro de Spam Mais Sofisticado

> **📌 Nota**
>
> Esta seção corresponde a *A More Sophisticated Spam Filter*, do capítulo 13 de Grus (2019).

Uma palavra só é pouco. Imagine agora um vocabulário inteiro, $w_1, \ldots, w_n$. Para levar isso para a teoria da probabilidade, escrevemos $X_i$ para o evento "a mensagem contém a palavra $w_i$". Também imaginamos que já temos, de algum jeito, uma estimativa $P(X_i \mid S)$ — a probabilidade de que uma mensagem de spam contenha a $i$-ésima palavra — e uma estimativa análoga $P(X_i \mid \lnot S)$ para mensagens legítimas.

### A suposição ingênua

A peça central do Naive Bayes é assumir — de forma grosseira — que as presenças (ou ausências) de cada palavra são independentes entre si, condicionadas a saber se a mensagem é spam ou não. Em termos matemáticos:

$$
P(X_1 = x_1, \ldots, X_n = x_n \mid S) = P(X_1 = x_1 \mid S) \times \cdots \times P(X_n = x_n \mid S)
$$

Aqui, cada $x_i$ vale 1 se a palavra $w_i$ aparece na mensagem e 0 caso contrário — e o fator correspondente muda de acordo: quando $x_i = 1$ o fator é $P(X_i \mid S)$, a probabilidade de ver a palavra; quando $x_i = 0$ o fator é $1 - P(X_i \mid S)$, a probabilidade de **não** vê-la. A ausência de uma palavra é informação tanto quanto a presença, e o modelo vai usar as duas.

Intuitivamente, a suposição significa que saber se uma mensagem de spam contém a palavra *bitcoin* não diz nada sobre se ela também contém *rolex*. É uma suposição extrema — daí o nome do método — e é fácil ver o quanto ela ignora: se o vocabulário fosse só *bitcoin* e *rolex*, e metade das mensagens de spam fosse "ganhe bitcoin" e a outra metade "rolex autêntico", o modelo estimaria

$$
P(X_1 = 1, X_2 = 1 \mid S) = P(X_1 = 1 \mid S)\, P(X_2 = 1 \mid S) = 0{,}5 \times 0{,}5 = 0{,}25
$$

para a probabilidade de uma mensagem de spam conter as duas palavras — mesmo sabendo, pela própria descrição do exemplo, que *bitcoin* e *rolex* nunca ocorrem juntas nesse conjunto de treino. A suposição está descartando, de propósito, exatamente a informação que tornaria a estimativa certa.

> **❗ Importante**
>
> Apesar de irrealista, essa suposição é o que torna o modelo **tratável**: em vez de estimar a probabilidade conjunta de todas as combinações possíveis de palavras — algo que exigiria dados demais para qualquer vocabulário razoável —, basta estimar $P(w_i \mid S)$ palavra por palavra e multiplicar. É uma troca deliberada de precisão por viabilidade, e funciona bem o suficiente na prática para ter sido, historicamente, usada de verdade em filtros de spam.

O mesmo raciocínio de Bayes usado no filtro de uma palavra dá a probabilidade de uma mensagem ser spam a partir de um vetor de presenças $x = (x_1, \ldots, x_n)$:

$$
P(S \mid X = x) = \frac{P(X = x \mid S)}{P(X = x \mid S) + P(X = x \mid \lnot S)}
$$

E a suposição de independência é o que permite calcular cada probabilidade do lado direito simplesmente multiplicando as estimativas individuais de cada palavra do vocabulário.

### O problema do *underflow*

Na prática, é melhor evitar multiplicar muitas probabilidades entre si, por causa de um problema chamado **underflow**. O menor `float` positivo normal fica em torno de $10^{-308}$ — e isso não é uma questão de precisão, é o alcance do expoente que a representação de ponto flutuante consegue guardar. Multiplicar, por exemplo, 300 fatores de 0,01 pediria algo perto de $10^{-600}$: não há expoente para isso, então o produto não fica impreciso — ele vira exatamente `0.0`, e leva a informação junto. Um vocabulário de milhares de palavras multiplica exatamente esse tipo de fator, milhares de vezes.

Lembrando da álgebra: $\log(ab) = \log a + \log b$ e $\exp(\log x) = x$. Então, em vez de calcular $p_1 \times \cdots \times p_n$ diretamente, calculamos o equivalente — mas mais amigável ao ponto flutuante:

$$
\exp\big(\log(p_1) + \cdots + \log(p_n)\big)
$$

Uma soma de logaritmos negativos nunca sofre *underflow* — um número muito negativo continua perfeitamente representável em `float`, ao contrário de um número muito próximo de zero pela direita, que eventualmente deixa de ser distinguível de zero. O `float` só corre risco na última etapa, o `exp` único, no fim de tudo — e só se a soma de logs já for extrema demais para qualquer probabilidade fazer sentido.

> **🟩 Exemplo**
>
> O exemplo a seguir é sintético, só para tornar o problema visível — não são números do classificador real. Imagine duas hipóteses concorrentes, cada uma resultando de multiplicar centenas de probabilidades pequenas. `np.full(300, 0.01)` cria um array de 300 posições, todas iguais a 0,01; `np.prod` multiplica todos os elementos de um array entre si — é a redução irmã do `np.sum` da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html), com produto no lugar da soma —, e `float()` traz o resultado de volta como número do Python, só para a saída sair limpa:

In [ ]:
import numpy as np

# 300 palavras "raras" sob a hipótese spam, com 1% de chance cada
probs_spam = np.full(300, 0.01)
# 300 palavras "raras" sob a hipótese ham, ligeiramente menos raras
probs_ham = np.full(300, 0.012)

float(np.prod(probs_spam)), float(np.prod(probs_ham))

In [ ]:
float(np.sum(np.log(probs_spam))), float(np.sum(np.log(probs_ham)))

> Multiplicando direto, as duas hipóteses colapsam para exatamente `0.0` — informação perdida, não dá mais para saber qual das duas era mais provável. Trabalhando em espaço de log, a diferença continua visível: a soma da hipótese `ham` é menos negativa, ou seja, mais provável. Repare que o remédio não é o `numpy`: `np.prod` sobre este array devolve o mesmo zero que um laço multiplicando os 300 fatores devolveria, porque o problema é do `float`, não de quem faz a conta. O que resolve é mudar de espaço — e é por isso que a comparação entre as duas hipóteses acontece **em log**, antes de qualquer `exp`. Neste exemplo, extremo de propósito, nem o `exp` final sobreviveria: $e^{-1381{,}55}$ também é `0.0`. No classificador de verdade, com o vocabulário de alguns milhares de palavras da [seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html), essas somas ficam entre $-105$ e $-7$ — e aí o `exp` único do fim devolve um número perfeitamente representável.

### Estimando as probabilidades por palavra

Falta só decidir como estimar $P(X_i \mid S)$ e $P(X_i \mid \lnot S)$ — a probabilidade de uma mensagem de spam (ou legítima) conter a palavra $w_i$. Com um conjunto de mensagens de treino já rotuladas, a primeira ideia é estimar $P(X_i \mid S)$ simplesmente como a fração das mensagens de spam que contêm $w_i$.

Isso causa um problema sério. Imagine que, no conjunto de treino, a palavra *data* só aparece em mensagens legítimas. Estimaríamos $P(\text{data} \mid S) = 0$ — e o classificador atribuiria probabilidade zero de spam a *qualquer* mensagem que contivesse "data", mesmo uma como "data sobre bitcoin grátis e rolex autêntico". Para evitar isso, usa-se **suavização** (*smoothing*).

Em particular, escolhemos um **pseudocontador** $k$ e estimamos a probabilidade de ver a $i$-ésima palavra numa mensagem de spam como (esta é a **suavização de Laplace**, também chamada de **regra da sucessão** — vale saber o nome para poder procurar o assunto por conta própria):

$$
P(X_i \mid S) = \frac{k + \text{número de spams contendo } w_i}{2k + \text{número de spams}}
$$

E de forma análoga para $P(X_i \mid \lnot S)$. Ou seja: ao calcular as probabilidades de spam para a $i$-ésima palavra, fingimos que também vimos $k$ spams adicionais contendo a palavra e $k$ spams adicionais sem ela — e o mesmo do lado das mensagens legítimas.

> **🟩 Exemplo**
>
> Se *data* ocorre em 0 de 98 mensagens de spam, e $k = 1$, a estimativa vira $P(\text{data} \mid S) = 1/100 = 0{,}01$ — pequena, refletindo que *data* é rara em spam, mas nunca exatamente zero. É o que permite ao classificador atribuir alguma probabilidade de spam, por menor que seja, a mensagens que contêm a palavra.
>
> Não existe um valor único e correto de $k$ — é um hiperparâmetro como outro qualquer, normalmente escolhido por validação. Ao longo deste capítulo você vai ver $k=1$ aqui, $k=0{,}5$ como padrão da classe na próxima seção, e ainda outros valores nos callouts de `scikit-learn` (que chama o mesmo parâmetro de `alpha`). Nenhum é "o certo": são escolhas diferentes de quão forte suavizar.

> **📌 Nota**
>
> Este é o mesmo problema, e a mesma solução, do [capítulo sobre extração de atributos](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html): um atributo do tipo sim-ou-não (a palavra apareceu ou não) precisa de uma estimativa que nunca trave em 0 ou em 1 diante de pouco dado. Um pseudocontador tem o mesmo espírito de um *prior* fraco — uma estimativa inicial, antes de olhar qualquer dado, que os dados corrigem aos poucos conforme chegam.

## Implementação

> **📌 Nota**
>
> Esta seção corresponde a *Implementation*, do capítulo 13 de Grus (2019).

Agora temos todas as peças para montar o classificador. Primeiro, uma função simples que transforma uma mensagem numa lista de palavras distintas: converte para minúsculas, usa uma expressão regular para extrair "palavras" (letras, números e apóstrofos), e usa um `set` para descartar as repetidas.

In [ ]:
from typing import Set
import re

def tokenize(text: str) -> Set[str]:
    text = text.lower()                          # Converte para minúsculas,
    all_words = re.findall("[a-z0-9']+", text)   # extrai as "palavras", e
    return set(all_words)                        # remove as duplicadas.

assert tokenize("Data Science is science") == {"data", "science", "is"}

O retorno é um `set` porque, para o modelo, só importa **se** uma palavra apareceu, não quantas vezes — a mesma escolha que sustenta a suavização da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-um-filtro-mais-sofisticado.html): cada palavra do vocabulário é um atributo sim-ou-não. `Set`, `NamedTuple` e `Counter`, todos apresentados no [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/index.html), trabalham juntos nesta seção — e o `Counter`, como você vai ver, só até o treino terminar.

> **❗ Importante — A decisão que este `set` já tomou**
>
> Guardar só **se** cada palavra apareceu, e não **quantas vezes**, não é um detalhe de implementação — é a escolha que decide qual das duas famílias de Naive Bayes para texto você está construindo. Este capítulo constrói a versão *Bernoulli*: cada palavra é presente ou ausente. A outra família, *multinomial*, guardaria contagens e perguntaria quantas vezes a palavra apareceu, não só se apareceu — é a variante mais comum na prática para texto corrido, e volta a aparecer no callout de `scikit-learn`, no fim desta seção.

Também precisamos de um tipo para representar cada mensagem de treino:

In [ ]:
from typing import NamedTuple

class Message(NamedTuple):
    text: str
    is_spam: bool

Como o classificador precisa acumular tokens, contagens e rótulos ao longo do treino, ele vira uma classe. O construtor recebe só um parâmetro — o pseudocontador $k$ a usar no cálculo das probabilidades — e inicializa um vocabulário vazio, dois arrays de contagens vazios (um por classe) e as contagens totais de mensagens de spam e de ham.

O método `train` percorre as mensagens de treino: primeiro incrementa a contagem de mensagens de spam ou de ham, depois tokeniza o texto e, para cada token, incrementa a contagem correspondente. Terminada a varredura, ele **consolida**: o vocabulário vira um array ordenado, e as duas contagens viram arrays de inteiros alinhados a ele. O método `_probabilities` é um método "privado" que calcula $P(\text{token} \mid \text{spam})$ e $P(\text{token} \mid \text{ham})$ — não para um token, mas para o vocabulário inteiro de uma vez, devolvendo dois arrays —, já com a suavização de pseudocontador da seção anterior embutida:

In [ ]:
from typing import Iterable, Tuple
from collections import Counter
import numpy as np

class NaiveBayesClassifier:
    def __init__(self, k: float = 0.5) -> None:
        self.k = k  # fator de suavização

        self.tokens: np.ndarray = np.array([], dtype=str)
        self.token_spam_counts: np.ndarray = np.array([], dtype=int)
        self.token_ham_counts: np.ndarray = np.array([], dtype=int)
        self.spam_messages = self.ham_messages = 0

    def train(self, messages: Iterable[Message]) -> None:
        spam_counts: Counter = Counter()
        ham_counts: Counter = Counter()

        for message in messages:
            # Incrementa as contagens de mensagens
            if message.is_spam:
                self.spam_messages += 1
            else:
                self.ham_messages += 1

            # Incrementa as contagens de palavras
            for token in tokenize(message.text):
                if message.is_spam:
                    spam_counts[token] += 1
                else:
                    ham_counts[token] += 1

        # A varredura acabou e o vocabulário está fechado: ele vira um array
        # ORDENADO, e as duas contagens viram arrays alinhados a ele — a
        # posição i é sempre o token self.tokens[i], em todos os três.
        self.tokens = np.array(sorted(spam_counts.keys() | ham_counts.keys()))
        self.token_spam_counts = np.array([spam_counts[t] for t in self.tokens],
                                          dtype=int)
        self.token_ham_counts = np.array([ham_counts[t] for t in self.tokens],
                                         dtype=int)

    def _probabilities(self) -> Tuple[np.ndarray, np.ndarray]:
        """P(token | spam) e P(token | ham) para todo o vocabulário, de uma vez."""
        p_token_spam = (self.token_spam_counts + self.k) / (self.spam_messages + 2 * self.k)
        p_token_ham = (self.token_ham_counts + self.k) / (self.ham_messages + 2 * self.k)

        return p_token_spam, p_token_ham

> **🟩 Exemplo**
>
> Repare que o `train` usa as duas estruturas, e em momentos diferentes.
>
> Enquanto a varredura acontece, o vocabulário ainda está crescendo: não se sabe quantas palavras existem, nem quais, e a cada mensagem pode aparecer uma nova. É a situação em que um `Counter` é a ferramenta certa — chave nova a qualquer momento, `spam_counts[token] += 1` sem perguntar se a chave já existia, e `spam_counts[t]` devolvendo 0 para quem nunca apareceu.
>
> Quando o laço termina, o vocabulário está fechado, e a partir daí o tamanho é conhecido e fixo. É a situação inversa, em que o array é a ferramenta certa: `sorted(...)` dá uma **ordem** ao vocabulário, e as duas contagens são construídas nessa mesma ordem. Daí em diante vale um contrato que o resto do capítulo usa o tempo todo: a posição $i$ é o token `self.tokens[i]`, em `tokens`, em `token_spam_counts` e em `token_ham_counts`. Uma conta feita sobre um dos três arrays vale, posição por posição, para os outros dois.
>
> As duas `Counter` são locais ao `train` — elas somem quando ele retorna. O que fica no modelo são os três arrays e os dois inteiros. Também é por isso que `train` é para ser chamado uma vez, sobre o conjunto de treino inteiro: ele não acumula sobre uma chamada anterior.

Falta o método que de fato decide: `predict`. Como prometido na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-um-filtro-mais-sofisticado.html), a ausência de uma palavra é informação tanto quanto a presença — então a conta abaixo passa por **todo** o vocabulário visto no treino, não só pelas palavras da mensagem, e soma $\log(1 - P(\text{token}\mid\text{spam}))$ para cada token ausente. Ela não passa palavra por palavra, porém: `np.isin(self.tokens, list(tokenize(text)))` compara o vocabulário inteiro com os tokens da mensagem de uma vez e devolve uma **máscara** — um booleano por posição, `True` onde aquele token aparece na mensagem. É a máscara booleana da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html), agora sobre palavras em vez de números; e é ela, e não o texto, a representação que o modelo enxerga: uma mensagem é um vetor de milhares de sim-ou-não. O `np.where` escolhe, posição por posição, entre o log de ver a palavra e o log de não vê-la, e o `np.sum` reduz tudo a um número. E, como discutido na seção anterior, somamos log-probabilidades em vez de multiplicar probabilidades pequenas, para não sofrer *underflow*:

In [ ]:
def predict(self, text: str) -> float:
    p_spam, p_ham = self._probabilities()

    # A máscara: True nas posições cujo token aparece na mensagem.
    presentes = np.isin(self.tokens, list(tokenize(text)))

    # Presente: soma log(p). Ausente: soma log(1 - p). Numa redução só.
    log_prob_if_spam = np.sum(np.where(presentes, np.log(p_spam), np.log(1 - p_spam)))
    log_prob_if_ham = np.sum(np.where(presentes, np.log(p_ham), np.log(1 - p_ham)))

    prob_if_spam = np.exp(log_prob_if_spam)
    prob_if_ham = np.exp(log_prob_if_ham)
    return float(prob_if_spam / (prob_if_spam + prob_if_ham))

NaiveBayesClassifier.predict = predict

E agora temos um classificador.

> **📌 Nota — Por que `predict` nasceu fora da classe**
>
> `predict` está escrito como uma função comum, de nível zero, e só na última linha vira método: `NaiveBayesClassifier.predict = predict`. Isso funciona porque, em Python, um método **é** só uma função guardada como atributo da classe — atribuir depois da classe já existir é tão válido quanto escrever `def predict(self, ...)` dentro do `class` desde o início.
>
> Escrever `predict` à parte dá à peça conceitualmente mais densa da classe a sua própria introdução, em vez de apresentar tudo de uma vez como uma parede de código. Mas a última linha é obrigatória: o corpo de uma classe é executado de uma vez só, quando o `class` termina. Nada escrito depois continua esse corpo — só a atribuição explícita anexa o método.

> **🔷 Conceito**
>
> Repare no que `predict` faz com **cada** token do vocabulário, não só com os que aparecem na mensagem: onde a máscara é `False`, o `np.where` escolhe $\log(1 - P(\text{token} \mid \text{spam}))$ — a log-probabilidade de *não* ver aquela palavra, cumprindo a promessa da seção anterior de que a ausência também é informação.
>
> É também o motivo pelo qual a máscara tem o tamanho do vocabulário inteiro visto no treino, e não o da mensagem. Uma mensagem curta contribui, mesmo assim, com um termo para cada uma das milhares de palavras do vocabulário — a maioria deles vindos do ramo `False`. Repare no preço disso em `numpy`: o `np.where` calcula os **dois** lados para todas as posições e só depois escolhe. É trabalho jogado fora, e ainda assim mais barato que decidir com um `if` por token, porque as duas contas correm em código compilado e o `if` correria no interpretador — medido sobre o vocabulário da [seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html), a máscara classifica uma mensagem em cerca de um décimo do tempo do laço.
>
> E há um caso que a máscara resolve calada: uma palavra que a mensagem tem e o treino nunca viu não está no vocabulário — logo, não tem posição, e não entra na conta. O modelo não tem nada a dizer sobre ela, e o `np.isin` simplesmente não a encontra.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O `scikit-learn` traz `BernoulliNB`, que implementa essencialmente o mesmo algoritmo — Naive Bayes sobre atributos binários (presença ou ausência de cada palavra):
>
> ```python
> from sklearn.feature_extraction.text import CountVectorizer
> from sklearn.naive_bayes import BernoulliNB
>
> vetorizador = CountVectorizer(binary=True)
> X = vetorizador.fit_transform(textos_treino)
>
> modelo = BernoulliNB(alpha=1.0)  # alpha é o k deste capítulo
> modelo.fit(X, y_treino)
> modelo.predict_proba(vetorizador.transform(textos_novos))
> ```
>
> O `CountVectorizer` faz a tokenização — de forma mais sofisticada que o `tokenize` acima, com opções para *n*-gramas, remoção de palavras comuns (*stop words*) e limites de frequência. O `alpha` do `BernoulliNB` é exatamente o pseudocontador $k$: o mesmo truque de suavização, só com outro nome. E nos bastidores, a biblioteca também soma log-probabilidades em vez de multiplicar — o mesmo cuidado com *underflow* que você acabou de implementar à mão. A diferença de forma é que ela classifica um lote inteiro de mensagens de uma vez, com uma multiplicação de matrizes onde o nosso `predict` faz uma máscara por mensagem; a conta por mensagem é a mesma.
>
> `BernoulliNB` é a metade da história. O `scikit-learn` também tem `MultinomialNB`, que guarda **contagens** em vez de presença/ausência — a família que o callout no início desta seção mencionou, e a escolha mais comum para texto corrido na prática. Trocar de um para o outro é trocar `binary=True` por `binary=False` no `CountVectorizer` (ou usar `TfidfVectorizer`, que pesa cada contagem pela raridade da palavra no corpus todo); o resto do código muda pouco. Duas coisas que a biblioteca esconde e que valem saber que existem: `fit_prior`, que por padrão estima $P(S)$ e $P(\lnot S)$ pelas proporções reais do treino em vez de assumir 50/50 como fizemos na [seção 10.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/01-um-filtro-de-spam-bem-burro.html); e `binarize`, que decide o limiar de contagem para "presente" quando você passa dados que ainda não são binários.
>
> Nada disso muda o que o modelo *é*: contar palavras por classe, suavizar, e combinar via Bayes assumindo independência.

## Testando o Modelo

> **📌 Nota**
>
> Esta seção corresponde a *Testing Our Model*, do capítulo 13 de Grus (2019).

Vamos garantir que o modelo funciona escrevendo alguns testes para ele — pequenos o bastante para que as contas caibam à mão. `tokenize`, `Message` e `NaiveBayesClassifier` não são redefinidos aqui: vêm de `scratch_np.naive_bayes`, o mesmo código que você escreveu na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/03-implementacao.html).

In [ ]:
import numpy as np
from scratch_np.naive_bayes import tokenize, Message, NaiveBayesClassifier

Primeiro, conferimos se as contagens saíram certas:

In [ ]:
messages = [Message("spam rules", is_spam=True),
            Message("ham rules", is_spam=False),
            Message("hello ham", is_spam=False)]

model = NaiveBayesClassifier(k=0.5)
model.train(messages)

assert np.array_equal(model.tokens, np.array(["ham", "hello", "rules", "spam"]))
assert model.spam_messages == 1
assert model.ham_messages == 2
assert np.array_equal(model.token_spam_counts, np.array([0, 0, 1, 1]))
assert np.array_equal(model.token_ham_counts, np.array([2, 1, 1, 0]))

{str(t): int(n) for t, n in zip(model.tokens, model.token_spam_counts)}

Três mensagens de treino, quatro palavras distintas no vocabulário — e o vocabulário está em ordem alfabética, porque foi assim que `train` o fechou. `rules` aparece numa mensagem de spam e numa de ham; `spam` só na de spam; `ham` e `hello` só nas de ham.

Vale olhar o que os `assert` das contagens exigem. `token_spam_counts` é `[0, 0, 1, 1]`, com quatro posições — uma para cada palavra do vocabulário, inclusive as duas que nunca apareceram em spam. Um dicionário guardaria `{"spam": 1, "rules": 1}` e omitiria os zeros; o array não tem essa opção, porque é o alinhamento com `model.tokens` que dá sentido a cada posição. A última linha é o jeito de ler os dois arrays juntos: `zip` casa token com contagem, e o `str()`/`int()` converte os escalares do `numpy` de volta para tipos do Python — o mesmo cuidado do `float()` da [seção 9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html), aqui só para a saída sair legível.

Agora fazemos uma previsão. Vamos refazer a lógica do Naive Bayes na mão — nas quatro posições do vocabulário, na ordem em que elas estão no array — e conferir que chegamos ao mesmo resultado que o método `predict`:

In [ ]:
text = "hello spam"

# Na ordem do vocabulário: ham, hello, rules, spam.
probs_if_spam = np.array([
    1 - (0 + 0.5) / (1 + 2 * 0.5),  # "ham"   (ausente)
    (0 + 0.5) / (1 + 2 * 0.5),      # "hello" (presente)
    1 - (1 + 0.5) / (1 + 2 * 0.5),  # "rules" (ausente)
    (1 + 0.5) / (1 + 2 * 0.5),      # "spam"  (presente)
])

probs_if_ham = np.array([
    1 - (2 + 0.5) / (2 + 2 * 0.5),  # "ham"   (ausente)
    (1 + 0.5) / (2 + 2 * 0.5),      # "hello" (presente)
    1 - (1 + 0.5) / (2 + 2 * 0.5),  # "rules" (ausente)
    (0 + 0.5) / (2 + 2 * 0.5),      # "spam"  (presente)
])

p_if_spam = np.exp(np.sum(np.log(probs_if_spam)))
p_if_ham = np.exp(np.sum(np.log(probs_if_ham)))

# Deve dar aproximadamente 0,83
assert np.isclose(model.predict(text), p_if_spam / (p_if_spam + p_if_ham))

model.predict(text)

O teste passa: `model.predict(text)` dá aproximadamente 0,835. Olhando as probabilidades de verdade, os dois termos que dominam esse número são que a mensagem contém *spam* (o que a nossa única mensagem de spam de treino também continha) e não contém *ham* (o que as duas mensagens de ham continham).

> **⚠️ Atenção — Um `assert` que compara `float` por igualdade exata**
>
> O Grus (2019) escreve a linha acima assim: `model.predict(text) == p_if_spam / (p_if_spam + p_if_ham)`. Não é uma comparação com tolerância — é igualdade **exata** de ponto flutuante entre dois cálculos que chegam ao mesmo valor por caminhos diferentes. E soma de ponto flutuante não é associativa: a ordem em que os termos são somados pode mudar o último bit do resultado, o que basta para `==` responder `False`.
>
> No código do Grus a ordem era o problema declarado: o `predict` dele soma percorrendo `self.tokens`, que lá é um `Set[str]`, e a ordem de iteração de um `set` depende do hash de cada string — que o Python randomiza por processo, a menos que a variável de ambiente `PYTHONHASHSEED` esteja fixada. O mesmo `assert`, rodado de novo, podia somar em outra ordem e estourar.
>
> Aqui o vocabulário é um array ordenado, e essa causa some: a ordem da soma é sempre a mesma. Isso **não** conserta o `assert` — só troca o motivo pelo qual ele é frágil. `np.sum` não soma da esquerda para a direita: ele usa soma pareada, que quebra o array em blocos e soma os blocos entre si, justamente para acumular menos erro. O agrupamento depende do **tamanho** do array, então a mesma soma, sobre um vocabulário maior, é feita em outra ordem — e o último bit muda de novo. Escrever `==` continua sendo apostar num detalhe de implementação que ninguém prometeu.
>
> A medida: as previsões desta versão em arrays e as da versão em dicionários do Grus (2019), sobre as 825 mensagens de teste da [seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html), coincidem até a décima segunda casa decimal e concordam em **todas** as 825 decisões — mas só **31 delas** são iguais bit a bit. Mesma matemática, ordens de soma diferentes.
>
> É a terceira vez que esta lição aparece, com fachadas diferentes: o `shuffle` que embaralha os índices de início dos lotes — não os pontos, deixando cada lote sempre com os mesmos exemplos — no [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/06-minibatch-e-estocastico.html), e a afirmação sobre correlação que só é reprodutível com a semente fixada, no [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html). Nos três casos, uma afirmação escrita como se fosse determinística, sobre algo que não é. Comparar `float` por igualdade é, especificamente, o erro que **você** vai cometer algum dia; reconhecê-lo aqui é o que faz você desconfiar da próxima vez.
>
> Por isso o `assert` acima usa `np.isclose`, que compara com tolerância — `math.isclose`, da biblioteca padrão, faz o mesmo para dois números soltos, e `np.allclose` faz para arrays inteiros. É a ferramenta certa sempre que dois cálculos de ponto flutuante deveriam chegar ao "mesmo" resultado por caminhos diferentes.

Agora vamos testar em dados de verdade.

## Usando o Modelo

> **📌 Nota**
>
> Esta seção corresponde a *Using Our Model*, do capítulo 13 de Grus (2019).

Um conjunto de dados popular (embora um tanto velho) é o [corpus público SpamAssassin](https://spamassassin.apache.org/old/publiccorpus/). Grus (2019) usa os arquivos com prefixo *20021010*, e traz um script que baixa e descompacta o conjunto inteiro:

```python
from io import BytesIO   # Para tratar bytes como um arquivo
import requests          # Para baixar os arquivos, que
import tarfile           # estão em formato .tar.bz.

BASE_URL = "https://spamassassin.apache.org/old/publiccorpus"
FILES = ["20021010_easy_ham.tar.bz2",
         "20021010_hard_ham.tar.bz2",
         "20021010_spam.tar.bz2"]

# É aqui que os dados vão parar,
# nos subdiretórios /spam, /easy_ham e /hard_ham.
OUTPUT_DIR = 'spam_data'

for filename in FILES:
    # Usa requests para baixar o conteúdo de cada URL.
    content = requests.get(f"{BASE_URL}/{filename}").content

    # Envolve os bytes em memória para usá-los como um "arquivo".
    fin = BytesIO(content)

    # E extrai todos os arquivos para o diretório de saída.
    with tarfile.open(fileobj=fin, mode='r:bz2') as tf:
        tf.extractall(OUTPUT_DIR)
```

Depois de baixar, você teria três pastas — *spam*, *easy_ham* e *hard_ham* —, cada uma com centenas de e-mails, um por arquivo. Para simplificar, olhamos só a linha de assunto de cada e-mail, identificada por começar com `"Subject:"`:

```python
import glob, re

# modifique o caminho para onde você colocou os arquivos
path = 'spam_data/*/*'

data: List[Message] = []

# glob.glob retorna todo nome de arquivo que casa com o caminho com curinga
for filename in glob.glob(path):
    is_spam = "ham" not in filename

    # Há alguns caracteres inválidos nos e-mails; errors='ignore'
    # pula-os em vez de levantar uma exceção.
    with open(filename, errors='ignore') as email_file:
        for line in email_file:
            if line.startswith("Subject:"):
                subject = line.lstrip("Subject: ")
                data.append(Message(subject, is_spam))
                break  # termina com este arquivo
```

> **❗ Importante**
>
> Os dois blocos de código acima **não são executados** aqui.
>
> O corpus SpamAssassin completo soma centenas de megabytes, espalhados por milhares de arquivos de e-mail. E o código logo acima descarta tudo, de cada arquivo, menos a linha `Subject:`. O resultado dessa varredura, já reduzido ao que o classificador de fato usa, está em `dados/spam-assuntos.csv`, com duas colunas, `assunto` e `is_spam` — é esse arquivo que carregamos a seguir.
>
> O código de download e o de varredura continuam à vista porque saber de onde o dado vem, e como alguém reduziria um e-mail inteiro a uma única linha, *é* parte da lição.

### Carregando os dados

In [ ]:
import csv
from typing import List
import numpy as np
from scratch_np.naive_bayes import Message

with open("dados/spam-assuntos.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    data: List[Message] = [Message(row["assunto"], row["is_spam"] == "True")
                            for row in reader]

total = len(data)
num_spam = sum(1 for m in data if m.is_spam)
num_ham = total - num_spam

total, num_spam, num_ham

3.300 assuntos, 2.800 ham e 500 spam.

> **⚠️ Atenção — Um conjunto desequilibrado, de propósito**
>
> 2.800 contra 500 não é acidente de amostragem — é como o corpus real se parece: a maior parte do que chega numa caixa de entrada não é spam. E é exatamente o cenário em que a acurácia engana, que a [seção sobre correção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) do Capítulo 8 ensinou com a piada do teste de leucemia: quando uma classe domina, um modelo pode acertar muito e não fazer nada de útil.
>
> Um classificador que respondesse "não é spam" para tudo, sem examinar uma palavra sequer, já sairia com acurácia alta só por causa dessa proporção — o equivalente ao teste "não tem leucemia para ninguém" daquele capítulo. Calculamos esse piso adiante, depois de separar treino e teste, nas mesmas métricas — não só na acurácia, que é justamente a que mais esconde — e o usamos como régua: é o mínimo que o classificador treinado precisa superar.

### Treinando e avaliando

Como sempre que há aleatoriedade, o gerador vem semeado, e ele é parâmetro da divisão — o `split_data` da [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) não tem valor padrão para ele:

In [ ]:
from scratch_np.machine_learning import split_data

rng = np.random.default_rng(0)
train_messages, test_messages = split_data(data, 0.75, rng)

treino_spam = sum(1 for m in train_messages if m.is_spam)

len(train_messages), len(test_messages), treino_spam

> **⚠️ Atenção — A semente não é opcional**
>
> `np.random.default_rng(0)` cria o gerador que o `split_data` usa para sortear a divisão — e ele é **parâmetro**, não estado global: a semente fica escrita aqui, ao lado da divisão que produziu. Sem ela, cada execução separaria outro conjunto de treino, e com ele viriam outra matriz de confusão, outras métricas e outra lista de palavras "spammy" — nenhuma das frases desta página continuaria batendo com a saída ao lado.
>
> O zero é o mesmo número que o Grus (2019) usa, mas o sorteio **não** é o mesmo: lá a semente alimenta o `random` da biblioteca padrão, aqui o gerador do `numpy`. Semente igual em geradores diferentes dá divisão diferente — e é por isso que os números desta seção não são os do livro-texto, embora o código seja o mesmo algoritmo.

2.475 mensagens de treino — 389 delas spam — e 825 de teste. O corte é determinístico: 75% de 3.300 são sempre 2.475, independentemente do sorteio; o que o sorteio decide é **quais** mensagens ficam de cada lado, e é dele que dependem todos os números daqui para baixo. Antes de treinar, calculamos o piso prometido no callout acima: como o classificador que respondesse sempre "não é spam" — tp = 0, fp = 0, sempre — se sairia nas métricas do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html), só olhando a composição do conjunto de teste.

In [ ]:
from scratch_np.machine_learning import accuracy, precision, recall, f1_score

e_spam = np.array([m.is_spam for m in test_messages])   # 825 booleanos

baseline_spam = int(e_spam.sum())        # True conta como 1
baseline_ham = int((~e_spam).sum())      # ~ inverte a máscara

baseline_accuracy = accuracy(0, 0, baseline_spam, baseline_ham)
baseline_recall = recall(0, 0, baseline_spam, baseline_ham)

baseline_spam, baseline_ham, baseline_accuracy, baseline_recall

714 das 825 mensagens de teste são ham, o que já entrega a acurácia do piso: 86,5% — para um classificador que nunca examina uma palavra. A revocação conta outra história: **0,0**. Dos 111 spams do conjunto de teste, esse classificador encontra nenhum.

Repare de passagem no que `e_spam` é: uma máscara de 825 booleanos, uma posição por mensagem de teste. `e_spam.sum()` conta os `True` porque `True` vale 1, e `~e_spam` é a máscara invertida — os ham. É com essas duas que a matriz de confusão vai ser montada daqui a pouco, sem contar nada à mão.

E a precisão nem chega a ser uma conta possível:

In [ ]:
try:
    precision(0, 0, baseline_spam, baseline_ham)
except ZeroDivisionError:
    print("precisão indefinida: 0 verdadeiros positivos sobre 0 previsões de spam")

`precision` divide `tp` por `tp + fp`, e as duas são zero — o classificador nunca aponta "spam", então não há nada para dividir. É, com dados reais, o classificador que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) usou para desmontar o teste do Luke: aquele que responde "não tem leucemia" para todo mundo e ainda assim acerta 98,6% das vezes. Note que não é o teste do Luke em si — aquele *aponta* alguém, então tem precisão de 1,4% e revocação de 0,5%, ambas definidas e ambas péssimas. Este aqui nunca aponta ninguém: acurácia enorme, revocação exatamente zero, precisão que não existe.

Agora treinamos o classificador propriamente dito e geramos previsões para o conjunto de teste. O laço percorre as 825 mensagens porque cada uma tem um texto próprio para tokenizar; a conta pesada — comparar a mensagem com o vocabulário inteiro e somar milhares de log-probabilidades — já é uma operação de array dentro do `predict`. `probabilidades` é o array das 825 saídas, e `previu_spam` é a máscara que o limiar de 0,5 produz sobre ele:

In [ ]:
from scratch_np.naive_bayes import NaiveBayesClassifier

model = NaiveBayesClassifier()
model.train(train_messages)

probabilidades = np.array([model.predict(m.text) for m in test_messages])

# Considera probabilidade > 0,5 como previsão de spam.
previu_spam = probabilidades > 0.5

len(model.tokens)

O treino fechou um vocabulário de 3.863 palavras. São 3.863 atributos sim-ou-não por mensagem — e o `predict` visita todos eles, para cada uma das 825 mensagens de teste, em cerca de um décimo de segundo no total.

In [ ]:
tp = int(np.sum(e_spam & previu_spam))     # é spam, previu spam
fp = int(np.sum(~e_spam & previu_spam))    # é ham,  previu spam
fn = int(np.sum(e_spam & ~previu_spam))    # é spam, previu ham
tn = int(np.sum(~e_spam & ~previu_spam))   # é ham,  previu ham

tp, fp, fn, tn

As quatro células da matriz de confusão saem das mesmas duas máscaras, combinadas: `&` é o "e" lógico posição por posição, e `~` inverte. `e_spam & previu_spam` é `True` exatamente onde a mensagem era spam **e** o modelo disse spam; somar essa máscara é contar esses casos. As quatro combinações cobrem as 825 mensagens sem sobra e sem repetição — $78 + 34 + 33 + 680 = 825$.

In [ ]:
accuracy(tp, fp, fn, tn), precision(tp, fp, fn, tn), recall(tp, fp, fn, tn), f1_score(tp, fp, fn, tn)

São 78 verdadeiros positivos, 34 falsos positivos, 680 verdadeiros negativos e 33 falsos negativos. `accuracy`, `precision`, `recall` e `f1_score` são as mesmas funções do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html). A acurácia sai em 91,9%, a precisão em 69,6%, a revocação em 70,3% e o F1 em 0,700.

> **🔷 Conceito**
>
> Cinco pontos de acurácia — 86,5% do piso contra 91,9% do classificador treinado — parecem pouco, e é exatamente aí que a acurácia engana, como a piada do Luke do Capítulo 8 avisou. As outras métricas contam outra história: o piso tinha revocação 0,0 e precisão indefinida; o classificador treinado tem revocação de 70,3% e F1 de 0,700 (o F1 do piso nem chega a existir — sem precisão, não há como calculá-lo). Não é uma melhora de cinco pontos. É a diferença entre um modelo e nada, com um número de acurácia enorme escondendo os dois lados dela — igual ao teste do Luke, só que aqui o modelo treinado também existe, e vence por uma distância que a acurácia sozinha nunca mostraria.
>
> Ainda assim, a precisão e a revocação do classificador treinado continuam moderadas, não excelentes. De cada 100 mensagens que o modelo aponta como spam, cerca de 30 são, na verdade, legítimas — e de cada 100 mensagens que realmente são spam, o modelo deixa passar cerca de 30. Para um classificador que olha só a linha de assunto, sem ver o corpo da mensagem, não é um resultado ruim; é o resultado de deliberadamente jogar fora quase toda a informação disponível.

Vale notar uma escolha que o modelo faz sem anunciar. `predict` devolve `prob_if_spam / (prob_if_spam + prob_if_ham)`, sem peso nenhum pela proporção real de spam no treino — o que equivale a assumir $P(\text{spam}) = P(\text{ham}) = 0{,}5$, a mesma simplificação da [seção 10.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/01-um-filtro-de-spam-bem-burro.html). Só que o treino tem 2.086 mensagens de ham para 389 de spam, não 50/50. Essa suposição embutida — e nunca corrigida por uma proporção real — explica parte dos 34 falsos positivos: o modelo empurra a decisão para "spam" com mais facilidade do que a proporção do treino justificaria. É também por onde o `scikit-learn` se afasta silenciosamente do nosso código, como o callout da seção anterior menciona: `BernoulliNB` usa `fit_prior=True` por padrão, estimando as duas probabilidades a partir das proporções de treino em vez de assumir 50/50.

Vale parar também num detalhe fácil de passar batido: `predict` devolve uma **probabilidade**, não um rótulo — e o `> 0.5` que usamos acima para transformar essa probabilidade em previsão é uma escolha nossa, não uma exigência do modelo. O [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) já avisou que, num filtro de spam, um falso positivo (e-mail importante que some no lixo) costuma custar mais caro que um falso negativo (propaganda que passa), e que por isso o limiar deveria pender para a precisão — um limiar mais alto que 0,5, que só chama de spam o que o modelo tem mais certeza. A troca é nessa direção: exigir mais confiança **derruba os 34 falsos positivos e paga com mais falsos negativos** que os 33 atuais. Você não ganha os dois lados; escolhe qual erro prefere cometer. Neste sorteio os dois erros estão quase empatados (34 contra 33), o que torna a escolha do limiar visivelmente uma troca: cada ponto que um lado ganha, o outro perde. É também o contraste mais limpo com o k-vizinhos do Capítulo 9: lá, `majority_vote` devolve um rótulo, ponto final, sem botão para girar depois. Aqui há um botão, e girá-lo é o que você de fato vai fazer ao usar isto.

> **📌 Nota**
>
> Grus (2019) observa, depois desses números: "presumivelmente faríamos melhor se olhássemos além da linha de assunto." É um lembrete de que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html) já tinha antecipado — o classificador Naive Bayes é adequado a atributos do tipo sim-ou-não, e a linha de assunto sozinha só entrega um subconjunto pequeno desses atributos. O corpo do e-mail inteiro daria muito mais palavras — muito mais atributos sim-ou-não — para o modelo condicionar sua decisão.

### O que o modelo aprendeu

O classificador também deixa inspecionar quais palavras são mais e menos indicativas de spam — basta ordenar o vocabulário pela probabilidade de spam dado o token, usando a mesma conta de Bayes de uma palavra só da [primeira seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/01-um-filtro-de-spam-bem-burro.html). Como `_probabilities()` devolve os dois arrays do vocabulário inteiro, a conta sai numa linha para as 3.863 palavras, e `np.argsort` — o mesmo da [seção 9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html) — devolve os índices que as ordenam. Como `model.tokens` está alinhado a esses arrays, indexá-lo por esses índices dá as palavras:

In [ ]:
# Não deveríamos chamar métodos "privados", mas é por uma boa causa.
p_spam, p_ham = model._probabilities()

# A conta de Bayes da seção 10.1, agora para as 3.863 palavras de uma vez.
p_spam_given_token = p_spam / (p_spam + p_ham)

ordem = np.argsort(p_spam_given_token, kind="stable")

print("mais spammy:", model.tokens[ordem[-10:]])
print("mais hammy:", model.tokens[ordem[:10]])

As palavras mais "spammy" incluem *rates*, *money*, *sale* e *mortgage* — o tipo de vocabulário publicitário que se espera de propaganda em massa —, além de *adv*, que vem do prefixo `ADV:` com que várias dessas mensagens abrem o assunto (a abreviação de *advertisement* que a legislação da época exigia de publicidade não solicitada), e *systemworks*, do "NORTON SYSTEMWORKS 2002 CLEARANCE SALE!" que se repete no corpus. As mais "hammy" incluem *spambayes*, *spamassassin*, *users*, *razor* e *apt* — vocabulário técnico de listas de discussão sobre software livre, que é de onde vem boa parte do ham deste corpus.

Duas dessas palavras não são palavras. *zzzz* é o nome fictício com que o corpus anonimizou o destinatário dos e-mails (`zzzz@example.com`) — e spam "personalizado", que repete o nome do destinatário no assunto, carrega esse marcador; *zzzzteana* é o nome, anonimizado do mesmo jeito, de uma das listas de discussão de onde veio o ham. Nos dois casos o modelo aprendeu a etiqueta com que o corpus foi montado, não a linguagem. E há um caso ainda mais local: *was*, *ouch* e *bliss* estão entre as dez mais hammy em boa parte porque um único assunto de thread — `Selling Wedded Bliss (was Re: Ouch...)`, nas variações de `Re:` e `Re[2]:` que uma discussão longa acumula — se repete dezenas de vezes no ham. Uma conversa só, com muitas respostas, vira evidência estatística.

Há empates: várias palavras aparecem em exatamente 7 mensagens de spam e em nenhuma de ham, e recebem o mesmo valor. `kind="stable"` faz o `argsort` manter, entre empatadas, a ordem em que elas já estavam — a alfabética do vocabulário. Sem isso, a ordenação continuaria correta, mas *quais* das empatadas entram nas dez primeiras dependeria do algoritmo de ordenação, e poderia mudar com a versão do `numpy`.

> **📌 Nota**
>
> Isso dá alguma confiança intuitiva de que o modelo está fazendo a coisa certa: ele não decorou rótulos, ele encontrou palavras que de fato diferenciam as duas classes neste corpus específico. "apt" ser hammy não é uma verdade universal sobre spam — é uma verdade sobre *estas* listas de discussão, onde e-mails técnicos legítimos falam de pacotes Debian com frequência. Um corpus de e-mails corporativos teria palavras hammy completamente diferentes.

### Como este modelo poderia melhorar

Grus (2019) fecha o capítulo com quatro sugestões de melhoria. Nenhuma delas é implementada aqui, mas todas valem como próximo passo para quem quiser levar o classificador adiante:

- **Mais dados de treino** — a saída mais óbvia, e a que costuma compensar mais que qualquer ajuste no algoritmo.
- Um limiar `min_count`, ignorando tokens vistos poucas vezes no treino. Palavras raras carregam pouca evidência estatística; incluí-las pode acrescentar ruído em vez de sinal.
- Um **stemmer**: uma função que reduz palavras aparentadas à mesma forma, para que `cheap` e `cheapest` contem como o mesmo token. O esboço mais ingênuo possível remove o "s" final:

  ```python
  def drop_final_s(word):
      return re.sub("s$", "", word)
  ```

  É um esboço de ideia, não uma peça do classificador. Um stemmer sério é bem mais delicado que remover um "s" final; o [Porter stemmer](https://tartarus.org/martin/PorterStemmer/), citado por Grus (2019), é o padrão da área.
- Atributos "fantasmas", como um token artificial `contains:number` que marcaria a presença de qualquer dígito na mensagem — uma forma de extrair sinal que `tokenize` sozinho não captura, sem sair do modelo de atributos sim-ou-não.

### Contraste com o Capítulo 9

Vale fechar comparando os dois classificadores construídos até aqui neste livro. O k-vizinhos mais próximos, do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html), não passa por nenhuma etapa de treino — o "modelo" é literalmente o conjunto de dados guardado, e toda a conta acontece na hora de classificar um ponto novo. Isso não quer dizer que ele não tenha nada para ajustar: a [seção 9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) escolheu $k=5$ à mão, e o texto de lá já avisava que "numa aplicação real criaríamos um conjunto de validação para escolher". O Naive Bayes é o oposto na etapa de treino: `train` percorre cada mensagem, acumula contagens e as consolida em `token_spam_counts` e `token_ham_counts`; depois de treinado, `predict` não olha mais para os dados brutos, só para esses dois arrays.

Vale comparar também o que cada modelo enxerga como "um exemplo". No [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html), a flor já chegava como quatro números, e a matriz `X` de forma `(150, 4)` era o dado inteiro — nada a construir. Aqui o dado é texto, e não existe matriz nenhuma: o que faz o papel do vetor de atributos é a máscara `presentes`, montada na hora, com 3.863 posições booleanas. Os dois modelos rodam sobre arrays; a diferença é que, num, o array **é** o dado, e no outro ele é uma representação que o modelo constrói a cada previsão.

Repare que os dois modelos têm, cada um, um hiperparâmetro chamado $k$ — e são duas coisas diferentes com o mesmo nome. O $k$ do Capítulo 9 é o número de vizinhos que votam; o $k$ deste capítulo é o pseudocontador que suaviza uma contagem de palavras (seção 10.2). Não há relação entre os dois além da letra — vale desarmar a confusão antes que ela aconteça.

Há um contraste mais interessante escondido nisso. A [seção 9.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-a-maldicao-da-dimensionalidade.html) gastou uma seção inteira mostrando que a distância — a única ferramenta do k-vizinhos — desmorona em dimensão alta, porque dois pontos só ficam próximos se estiverem próximos em **todas** as dimensões ao mesmo tempo. O vocabulário deste capítulo tem milhares de palavras — milhares de dimensões, pela mesma contagem daquela seção — e o Naive Bayes não sente nada disso. A suposição de independência da [seção 10.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-um-filtro-mais-sofisticado.html) é exatamente o que evita o problema: em vez de medir distância num espaço de milhares de dimensões, o modelo soma milhares de log-probabilidades unidimensionais, uma por palavra — e as 3.863 entram numa redução só. É geometria trocada por contagem — e é por isso que este classificador continua funcionando exatamente onde o k-vizinhos desmoronaria.

> **🔷 Conceito**
>
> Essa é uma diferença que vale generalizar. Modelos "preguiçosos" (*lazy*), como o k-vizinhos, adiam todo o trabalho para a hora da previsão e precisam guardar o conjunto de treino inteiro. Modelos que passam por um treino de fato, como o Naive Bayes, fazem o trabalho uma vez, guardam um resumo compacto — aqui, um vocabulário e dois arrays de contagens alinhados a ele — e descartam os dados brutos. O compromisso troca tempo de treino por tempo de previsão, e memória de treino por memória de previsão.

O padrão que importa carregar adiante não é o Naive Bayes em si, mas essa postura: comprimir o treino em parâmetros pequenos e jogar fora o resto. Os capítulos de regressão, a partir daqui, fazem exatamente isso — só que os parâmetros deixam de ser contagens de palavras e passam a ser coeficientes ajustados por gradiente descendente.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O mesmo experimento, de ponta a ponta, com a biblioteca:
>
> ```python
> from sklearn.feature_extraction.text import CountVectorizer
> from sklearn.model_selection import train_test_split
> from sklearn.naive_bayes import BernoulliNB
> from sklearn.metrics import confusion_matrix, precision_score, recall_score
>
> textos = [m.text for m in data]
> rotulos = [m.is_spam for m in data]
>
> X_treino, X_teste, y_treino, y_teste = train_test_split(
>     textos, rotulos, test_size=0.25, random_state=0
> )
>
> vetorizador = CountVectorizer(binary=True, token_pattern=r"[a-z0-9']+", lowercase=True)
> X_treino_vec = vetorizador.fit_transform(X_treino)
> X_teste_vec = vetorizador.transform(X_teste)
>
> modelo = BernoulliNB(alpha=0.5).fit(X_treino_vec, y_treino)
> previsto = modelo.predict(X_teste_vec)
>
> confusion_matrix(y_teste, previsto)
> precision_score(y_teste, previsto), recall_score(y_teste, previsto)
> ```
>
> Os números não saem idênticos aos nossos, por três motivos: o `train_test_split` embaralha de outro jeito; o `CountVectorizer` tokeniza com outra expressão regular por padrão (seu `token_pattern` default exige ao menos duas letras, o que descartaria tokens como o `"a"` que o nosso `tokenize` mantém); e, o maior deles, `fit_prior=True` por padrão — a biblioteca estima $P(\text{spam})$ e $P(\text{ham})$ pelas proporções reais do treino (389 contra 2.086 no nosso sorteio), não pelo 50/50 que o nosso `predict` assume desde a seção 10.1. Passar `fit_prior=False` reproduziria essa suposição, para quem quiser comparar as duas versões lado a lado. Mesmo com essas diferenças, a ideia é igual de ponta a ponta: contar palavras por classe, suavizar com um pseudocontador, combinar por Bayes assumindo independência.
>
> O que a biblioteca não faz por você é a parte mais importante desta seção: dizer *quais* palavras pesaram na decisão. `BernoulliNB` guarda o equivalente a `token_spam_counts` e `token_ham_counts` em `feature_log_prob_` — um array alinhado ao vocabulário, exatamente como os nossos, com `vetorizador.get_feature_names_out()` no lugar de `model.tokens`. Extrair dali as palavras mais "spammy" é o mesmo `argsort` que você acabou de escrever: a introspecção não vem de graça só por trocar de biblioteca. (E, como a seção de implementação mencionou, `BernoulliNB` é só metade da história — `MultinomialNB`, que conta em vez de marcar presença, é a escolha mais comum para texto corrido fora deste capítulo.)

## Leituras adicionais

A seção "For Further Exploration" do capítulo 13 de Grus (2019) sugere:

- Os artigos de Paul Graham ["A Plan for Spam"](http://www.paulgraham.com/spam.html) e ["Better Bayesian Filtering"](http://www.paulgraham.com/better.html), que deram origem histórica a boa parte das técnicas deste capítulo e trazem mais intuição sobre a construção de filtros de spam.
- O `scikit-learn` traz `BernoulliNB`, que implementa essencialmente o mesmo algoritmo Naive Bayes construído aqui, além de outras variações do modelo — o callout de fechamento da [seção de implementação](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/03-implementacao.html) mostra como usá-lo.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.